# 階段 6：Error analysis 與評分函式

對應文件：`docs/06-stage-evals-observability.md`。評測要能在 CI 跑，所以正式版本放 `evals/*.py`；這份 notebook 是工作台：

1. **Error analysis**：一筆一筆讀 trace、寫筆記、歸類計數（這件事本來就適合在 notebook 做）
2. **原型**：先在這裡把題目樣板、評分函式調好，附自我檢查，再搬進 `evals/`

前置：步驟 1 的 tracing（Langfuse 或 OpenTelemetry）已接好，或至少 agent 每題都會寫一行 JSONL log。

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import random
import re
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent  # kernel 的工作目錄是 notebooks/，往上一層才是專案根目錄

## 步驟 1：Error analysis

**先看資料再定指標。** 準備約 50 題多樣的問題（seed 題、你自己會問的真實問題、刁鑽題、應該拒答的題），跑過 agent，把每題的 trace 匯出成 `data/traces.jsonl`（每行至少有 `id`、`question`、`answer`，最好附上工具呼叫與 SQL）。

### 1.1 Open coding：一筆一筆讀，寫一兩句「哪裡不對」

先不要分類。下一格每次顯示一筆，你把筆記寫進 `NOTES`。

In [ ]:
TRACES = ROOT / "data" / "traces.jsonl"
traces = [json.loads(l) for l in TRACES.read_text().splitlines()] if TRACES.exists() else []
NOTES: dict[str, str] = {}   # trace id -> 你的筆記；沒問題就寫 "ok"

def show_next():
    todo = [t for t in traces if t["id"] not in NOTES]
    if not todo:
        print("全部讀完"); return
    t = todo[0]
    print(f"[{len(NOTES) + 1}/{len(traces)}] id={t['id']}\nQ: {t['question']}\n")
    print(json.dumps({k: v for k, v in t.items() if k not in ("id", "question")}, ensure_ascii=False, indent=1)[:3000])

show_next()

In [ ]:
# 讀完一筆就在這裡寫筆記，再回上一格跑 show_next()
# NOTES["q-001"] = "問最近 30 天，SQL 沒加時間條件"

### 1.2 Axial coding：歸成 5 到 8 種失敗類型並計數

例如：選錯工具、SQL 條件漏掉、編出 CVE、引用不支持說法、該拒答沒拒答。把每筆筆記對到一個類型，算次數、依次數與嚴重度排序，決定後面先量哪些指標。結果存成 `evals/error_analysis.md`。

In [ ]:
CATEGORY: dict[str, str] = {}   # trace id -> 失敗類型；TODO：讀完筆記後自己填

counts = Counter(c for c in CATEGORY.values() if c != "ok")
for cat, n in counts.most_common():
    print(f"{cat:20} {n:3}  {n / max(len(CATEGORY), 1):.0%}")

## 步驟 2：題目產生器原型

漏洞資料最大的好處：**標準答案能用程式從資料庫算出來**。題目由樣板加參數產生，答案跑 SQL 算出。評測一律對固定的資料快照跑，才分得出是模型退步還是資料變了。

下面的範例欄位沿用階段 1 的 DuckDB schema，請換成你實際的 schema。

In [ ]:
import duckdb

TEMPLATES = [
    {
        "id": "product_kev_cvss",
        "questions": [
            "影響 {product} 而且已經被利用、CVSS {cvss} 以上的漏洞有哪些？",
            "{product} 有哪些在 KEV 裡的高風險 CVE（CVSS ≥ {cvss}）？",
        ],
        "sql": """SELECT c.cve_id FROM cve c JOIN kev k USING (cve_id)
                  WHERE c.products ILIKE $pattern AND c.cvss_score >= $cvss""",
        "expected_tool": "search_vulns",
    },
    # TODO：再寫 7 到 11 種樣板：查單一 CVE、EPSS 排序、時間區間、各 severity 計數、CWE 篩選、組合條件...
]

def make_case(tpl: dict, params: dict, conn) -> dict:
    all_params = {**params, "pattern": f"%{params.get('product', '')}%"}
    sql_params = {k: v for k, v in all_params.items() if f"${k}" in tpl["sql"]}  # DuckDB 不接受多餘的參數
    gold = sorted(r[0] for r in conn.execute(tpl["sql"], sql_params).fetchall())
    q = random.choice(tpl["questions"]).format(**params)
    return {"question": q, "gold_sql": tpl["sql"], "params": params, "gold_result": gold,
            "expected_tool": tpl["expected_tool"], "template": tpl["id"]}

# 用記憶體內的小資料庫自我檢查
_db = duckdb.connect()
_db.execute("CREATE TABLE cve (cve_id TEXT, products TEXT, cvss_score DOUBLE)")
_db.execute("CREATE TABLE kev (cve_id TEXT)")
_db.execute("INSERT INTO cve VALUES ('CVE-2024-0001', 'nginx', 9.8), ('CVE-2024-0002', 'nginx', 5.0), ('CVE-2024-0003', 'openssl', 9.1)")
_db.execute("INSERT INTO kev VALUES ('CVE-2024-0001'), ('CVE-2024-0002'), ('CVE-2024-0003')")
_case = make_case(TEMPLATES[0], {"product": "nginx", "cvss": 7}, _db)
assert _case["gold_result"] == ["CVE-2024-0001"], _case
print(_case["question"])

### 2.1 人工抽查

自動產生的標準答案也會錯。每種樣板隨機抽 5 題看「題目合不合理、答案對不對」，有歧義的標成 `ambiguous`，統計時分開算。

In [ ]:
def sample_for_review(cases: list[dict], per_template: int = 5, seed: int = 0) -> list[dict]:
    rng = random.Random(seed)
    by_tpl: dict[str, list] = {}
    for c in cases:
        by_tpl.setdefault(c["template"], []).append(c)
    return [c for group in by_tpl.values() for c in rng.sample(group, min(per_template, len(group)))]

## 步驟 3：評分函式

### 3.1 SQL 正確率：比執行結果，不比 SQL 字串

同一個問題有很多種正確 SQL。比「執行結果」是否相同（execution accuracy）；題目有要求排序（「前 5 名」）才比順序；欄位別名與欄位順序忽略。

In [ ]:
def same_result(pred_rows: list[tuple], gold_rows: list[tuple], ordered: bool = False) -> bool:
    norm = lambda rows: [tuple(sorted(map(str, r))) for r in rows]   # 忽略欄位順序
    p, g = norm(pred_rows), norm(gold_rows)
    return p == g if ordered else sorted(p) == sorted(g)

assert same_result([("a", 1), ("b", 2)], [(2, "b"), (1, "a")])
assert not same_result([("a",), ("b",)], [("b",), ("a",)], ordered=True)
assert not same_result([("a",)], [("a",), ("a",)])   # 重複列要算

### 3.2 幻覺率：回答裡的 CVE 與分數都要出自本次查詢結果

In [ ]:
CVE_RE = re.compile(r"CVE-\d{4}-\d{4,}")

def hallucinations(answer: str, tool_results: list[dict], tol: float = 0.05) -> list[str]:
    """回傳幻覺清單。tool_results 每筆至少有 cve_id，可選 cvss_score、epss。"""
    known = {r["cve_id"]: r for r in tool_results}
    problems = [f"不存在於查詢結果：{c}" for c in sorted(set(CVE_RE.findall(answer))) if c not in known]
    # 檢查「CVE 編號後面緊接著出現的分數」，允許四捨五入誤差
    for cve_id, num in re.findall(r"(CVE-\d{4}-\d{4,})[^0-9\n]{0,30}?(\d+(?:\.\d+)?)", answer):
        r = known.get(cve_id)
        if r is None:
            continue
        candidates = [v for v in (r.get("cvss_score"), r.get("epss")) if v is not None]
        if candidates and all(abs(float(num) - v) > tol for v in candidates):
            problems.append(f"{cve_id} 的分數 {num} 對不上資料")
    return problems

_rows = [{"cve_id": "CVE-2024-0001", "cvss_score": 9.8, "epss": 0.92}]
assert hallucinations("CVE-2024-0001 的 CVSS 9.8", _rows) == []
assert hallucinations("另外 CVE-2023-9999 也很危險", _rows) == ["不存在於查詢結果：CVE-2023-9999"]
assert hallucinations("CVE-2024-0001 的 CVSS 7.5", _rows) == ["CVE-2024-0001 的分數 7.5 對不上資料"]

這個數字比對是粗略版：它會漏掉離 CVE 編號很遠的數字，也可能把年份以外的其他數字誤判。跑過一輪後，拿誤判的例子回來改它，這也是 error analysis。

### 3.3 工具選擇的混淆矩陣

In [ ]:
def confusion(results: list[dict]) -> Counter:
    """results 每筆有 expected_tool、chosen_tool。回傳 (預期, 實際) -> 次數"""
    return Counter((r["expected_tool"], r.get("chosen_tool") or "(沒呼叫)") for r in results)

_m = confusion([{"expected_tool": "cve_lookup", "chosen_tool": "run_sql"},
                {"expected_tool": "cve_lookup", "chosen_tool": "cve_lookup"}])
assert _m[("cve_lookup", "run_sql")] == 1

## 步驟 4：LLM-as-judge 的校正

評審模型對每個標準只回 pass / fail。用人工標註的驗證組算：

- **TPR**：真正失敗的答案裡，judge 抓到幾成
- **TNR**：真正好的答案裡，judge 放行幾成

兩個都要看。只看整體一致率會被「大部分答案都 pass」的資料騙。

In [ ]:
def judge_agreement(human: list[bool], judge: list[bool]) -> dict:
    """True 代表 pass。把 fail 當作要抓的「陽性」。"""
    pairs = list(zip(human, judge))
    fails = [j for h, j in pairs if not h]
    passes = [j for h, j in pairs if h]
    return {"TPR": sum(not j for j in fails) / len(fails) if fails else None,
            "TNR": sum(j for j in passes) / len(passes) if passes else None,
            "n": len(pairs)}

assert judge_agreement([False, False, True, True], [False, True, True, True]) == {"TPR": 0.5, "TNR": 1.0, "n": 4}

**TODO**：自己人工標註 30 到 50 題 pass / fail，分成調整用和驗證用兩組。用調整組改評分標準，用驗證組算一次 TPR / TNR，記下調整過程。

這些函式調好之後搬進 `evals/run.py`，接著照 `docs/06-stage-evals-observability.md` 的步驟 5 放進 GitHub Actions。